In [2]:
# =============================
# INSTALL LIBRARIES
# =============================
!pip install -q transformers datasets jiwer accelerate huggingface_hub
!pip install -q matplotlib
!pip install -q evaluate

# =============================
# IMPORTS
# =============================
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)
from huggingface_hub import login, upload_folder
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import evaluate
from datetime import datetime
import json

# =============================
# LOGIN
# =============================

# Load HF token from environment (.env)
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found in environment. Add it to .env or set environment variable.")

login(hf_token)

# =============================
# LOAD YOUR SAVED MODEL (BASE CHECKPOINT)
# =============================
model_id = "eshangj/TrOCR-Sinhala-finetuned"

processor = TrOCRProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

# Check for GPU availability and move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded on device: {device}")

# =============================
# DATA PATH
# =============================
DATA_DIR = os.getenv("DATA_DIR", "/datasets/train_images")
CSV_PATH = os.getenv("CSV_PATH", "/datasets/metadata.csv")

class SinhalaDataset(torch.utils.data.Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df.iloc[idx]['file_name']
        text = self.df.iloc[idx]['text']

        # Ensure pathing is correct
        img_path = os.path.join(DATA_DIR, file_name)
        image = Image.open(img_path).convert("RGB")

        # Process image only
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        # Tokenize text separately with fixed padding
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        # Replace pad token with -100
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# =============================
# LOAD DATA
# =============================
# CSV_PATH should point to a CSV with columns: file_name, text, is_handwritten
# Keep only handwritten samples where is_handwritten is true.
df = pd.read_csv(CSV_PATH)
if "is_handwritten" not in df.columns:
    raise ValueError("CSV is missing required column: is_handwritten")

is_handwritten_map = {"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False}
df["is_handwritten"] = (
    df["is_handwritten"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(is_handwritten_map)
)
df = df[df["is_handwritten"] == True].reset_index(drop=True)

if df.empty:
    raise ValueError("No handwritten samples found after filtering is_handwritten == true.")

# Create train / eval split (10% for eval)
val_df = df.sample(frac=0.1, random_state=42)
train_df = df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_dataset = SinhalaDataset(train_df, processor)
eval_dataset = SinhalaDataset(val_df, processor)

# Load metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# postprocess helper
def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

# compute_metrics for Seq2SeqTrainer (expects generated preds and label ids)
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    # decode generated token ids to strings
    decoded_preds = processor.batch_decode(preds, skip_special_tokens=True)

    # replace -100 in labels, then decode
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"cer": cer, "wer": wer}

# =============================
# RUN LOGGING
# =============================
LOG_PATH = os.getenv("LOG_PATH", "./training_logs/training_metrics.jsonl")

def append_run_log(log_entry):
    log_dir = os.path.dirname(LOG_PATH)
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)

    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

def verify_saved_artifacts(save_dir):
    required_files = ["config.json"]
    weight_files = ["model.safetensors", "pytorch_model.bin"]
    processor_files = ["preprocessor_config.json", "processor_config.json"]

    existing_files = set(os.listdir(save_dir))
    missing_required = [file_name for file_name in required_files if file_name not in existing_files]
    if not any(file_name in existing_files for file_name in weight_files):
        missing_required.append("model weights (model.safetensors or pytorch_model.bin)")
    if not any(file_name in existing_files for file_name in processor_files):
        missing_required.append("processor config (preprocessor_config.json or processor_config.json)")

    if missing_required:
        raise FileNotFoundError(f"Missing saved artifacts in {save_dir}: {missing_required}")

    return sorted(existing_files)

def get_best_epoch_metrics(log_history):
    epoch_entries = [entry for entry in log_history if "eval_loss" in entry]
    if not epoch_entries:
        return None

    best_entry = min(epoch_entries, key=lambda entry: entry["eval_loss"])
    return {
        "epoch": int(best_entry["epoch"]),
        "eval_loss": float(best_entry["eval_loss"]),
        "eval_cer": float(best_entry["eval_cer"]) if "eval_cer" in best_entry else None,
        "eval_wer": float(best_entry["eval_wer"]) if "eval_wer" in best_entry else None,
        "step": int(best_entry["step"]) if "step" in best_entry else None,
    }

# =============================
# TRAINING SETTINGS
# =============================
train_loss_values = []

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    learning_rate=2e-5,  # smaller LR for continued training
    fp16=True,
    save_strategy="epoch",
    logging_steps=20,
    eval_strategy="epoch",            # enable evaluation each epoch
    predict_with_generate=True,         # run generation for metrics
    report_to="tensorboard",  # Enable TensorBoard for tracking
)

# Define a custom trainer to capture the loss values and plot them
class CustomTrainer(Seq2SeqTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_values = []

    def log(self, logs, *args, **kwargs):
        super().log(logs, *args, **kwargs)
        if 'loss' in logs:
            self.loss_values.append(logs['loss'])

    def plot_training_loss(self):
        plt.figure(figsize=(10, 5))
        plt.plot(self.loss_values)
        plt.xlabel('Steps (Logging Frequency)')
        plt.ylabel('Loss')
        plt.title('Sinhala OCR Training Loss')
        plt.show()

# Create trainer instance
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,)
# =============================
# START TRAINING
# =============================
trainer.train()

# Plot the training loss graph after training
trainer.plot_training_loss()

# =============================
# SAVE MODEL (v3)
# =============================
# Use a local, configurable default path for saving so it works on your machine
save_path = os.getenv("SAVE_PATH", "./sinhala_model_v3")
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
processor.save_pretrained(save_path)
saved_files = verify_saved_artifacts(save_path)
print(f"Saved model artifacts: {saved_files}")

print("✅ Continued training completed successfully.")


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Model loaded on device: cpu


c:\Users\prasa\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Cer,Wer
1,2.124181,1.894334,0.392086,0.603774


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ModuleNotFoundError: No module named 'torch.utils.serialization'

In [ ]:
# Save the model to a specific path (portable for local runs)
save_path = os.getenv("SAVE_PATH", "./sinhala_model_v3")
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
processor.save_pretrained(save_path)
saved_files = verify_saved_artifacts(save_path)
print(f"Saved model artifacts: {saved_files}")

print(f"✅ Model saved at {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved at /kaggle/working/sinhala_model_v2


In [ ]:
# =============================
# PUSH TO HUGGING FACE
# =============================
repo_name = os.getenv("HF_REPO_NAME", "hasindu-k/sinhala-handwritten-notes-v3")

# Compute final predictions on eval set to get CER/WER numbers
print("Computing final evaluation predictions and metrics...")
preds_output = trainer.predict(eval_dataset, predict_with_generate=True)
raw_preds = preds_output.predictions
if isinstance(raw_preds, tuple):
    raw_preds = raw_preds[0]

decoded_preds = processor.batch_decode(raw_preds, skip_special_tokens=True)
label_ids = preds_output.label_ids
label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
decoded_labels = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
final_cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
final_wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)

print(f"Final CER: {final_cer}")
print(f"Final WER: {final_wer}")

best_epoch_metrics = get_best_epoch_metrics(trainer.state.log_history)
if best_epoch_metrics is not None:
    print(f"Best Epoch Metrics: {best_epoch_metrics}")
else:
    print("Best Epoch Metrics: not available")

run_log = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "model_id": model_id,
    "save_path": os.getenv("SAVE_PATH", "./sinhala_model_v3"),
    "repo_name": repo_name,
    "config": {
        "output_dir": training_args.output_dir,
        "train_batch_size": training_args.per_device_train_batch_size,
        "eval_batch_size": training_args.per_device_eval_batch_size,
        "num_train_epochs": training_args.num_train_epochs,
        "learning_rate": training_args.learning_rate,
        "fp16": training_args.fp16,
        "save_strategy": training_args.save_strategy.value if hasattr(training_args.save_strategy, "value") else str(training_args.save_strategy),
        "logging_steps": training_args.logging_steps,
        "eval_strategy": training_args.eval_strategy.value if hasattr(training_args.eval_strategy, "value") else str(training_args.eval_strategy),
        "predict_with_generate": training_args.predict_with_generate,
        "report_to": training_args.report_to,
        "max_length": 64,
        "eval_split_frac": 0.1,
        "random_state": 42,
    },
    "best_epoch_metrics": best_epoch_metrics,
    "final_cer": float(final_cer),
    "final_wer": float(final_wer),
    "logged_loss_count": len(trainer.loss_values),
    "first_loss": float(trainer.loss_values[0]) if trainer.loss_values else None,
    "last_loss": float(trainer.loss_values[-1]) if trainer.loss_values else None,
    "min_loss": float(min(trainer.loss_values)) if trainer.loss_values else None,
    "loss_values": [float(loss) for loss in trainer.loss_values],
}
append_run_log(run_log)
print(f"Training metrics logged to {LOG_PATH}")

# Upload the full saved artifact folder and run log to the Hub
upload_folder(
    repo_id=repo_name,
    folder_path=save_path,
    path_in_repo=".",
    commit_message="Upload trained model artifacts",
)
upload_folder(
    repo_id=repo_name,
    folder_path=os.path.dirname(LOG_PATH) or ".",
    path_in_repo="training_logs",
    allow_patterns=[os.path.basename(LOG_PATH)],
    commit_message="Upload training metrics log",
)

print(f"🚀 Model pushed to https://huggingface.co/{repo_name}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🚀 Model pushed successfully to Hugging Face!
